# Table 4 — CP at Timepoint 1 predicting concurrent (TP1) outcomes

**v2 of the public analysis notebooks.** Reproduces the manuscript table exactly.

### What changed from v1
The v1 notebook re-standardized canonical proportion (CP) *inside each outcome's
own complete-case subset* (e.g. `cp_z = (cp - cp.mean()) / cp.std()` after
dropping rows missing that outcome). Because every outcome drops a different set
of rows, that produced a different CP standard deviation per model.

This v2 standardizes CP **once over the full cohort** (every child with a CP
value), computed in code (see `load_master`). The scaled outcome used by the
scaled-outcome model is z-scored over each outcome's own complete-case sample.

For the **Timepoint 1 concurrent** master this also corrects a bug: a pre-stored
`canonical_proportion_scaled` column there had been computed over only the 117
GFTA-complete children, which silently dropped the 6 EVT-2 children who lacked
GFTA (EVT-2 model N = 116 instead of 122) and shifted the concurrent CP betas.
Computing the scaling in code fixes both. (R², log-likelihoods, the LRT χ², and
p-values are invariant to the CP scaling base.)

### Models fit per outcome
```
Baseline (unscaled) : OLS  outcome        ~ age + gender + maternal_ed
Expanded (unscaled) : OLS  outcome        ~ age + gender + maternal_ed + CP_scaled
Expanded (scaled)   : WLS  outcome_scaled ~ age + gender + maternal_ed + CP_scaled   (weights)
Weighted            : WLS  outcome        ~ age + gender + maternal_ed + CP_scaled   (weights)
```
Weights = (canonical + non-canonical clips) normalized to mean 1. All four models
for a given outcome are fit on the same complete-case sample so the
likelihood-ratio tests are valid.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

# ----------------------------------------------------------------------------
# v2 change (see markdown above): canonical proportion (CP) is z-scored ONCE over
# the full cohort (every child with a CP value), computed here in code rather than
# read from a pre-stored column. The v1 notebooks re-standardized CP inside each
# outcome's complete-case subset, which shifted the CP coefficients and -- for the
# Timepoint 1 concurrent EVT-2 model -- silently dropped 6 children (N=116 not 122).
# R^2, log-likelihoods, the LRT chi-square, and p-values are invariant to the CP
# scaling base; only the CP beta/SE shift (and, for EVT-2 concurrent, the sample).
# ----------------------------------------------------------------------------

COVARIATES = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
CP = "cp_scaled"   # cohort-standardized CP, computed in load_master()


def to_gender_dummy(x):
    """Female = 1, Male = 0. Accepts 'M'/'F', 'male'/'female', or numeric 0/1."""
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return int(v)
    except (TypeError, ValueError):
        pass
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan


def load_master(path):
    df = pd.read_csv(path)
    df["gender_dummy"] = df["gender"].apply(to_gender_dummy)
    # num_clips = canonical + non-canonical clips (the CP denominator / weight).
    # Stored in the longitudinal masters; the concurrent master keeps per-class
    # counts in columns "0".."4" (0 = canonical, 1 = non-canonical).
    if "num_clips" not in df.columns:
        df["num_clips"] = df[["0", "1"]].astype(float).sum(axis=1)
    # Standardize CP ONCE over the full cohort (ddof=0), computed in code.
    cohort = df["canonical_proportion"].dropna()
    df["cp_scaled"] = (df["canonical_proportion"] - cohort.mean()) / cohort.std(ddof=0)
    return df


def _lrt(restricted, full):
    stat = 2.0 * (full.llf - restricted.llf)
    ddf = int(full.df_model - restricted.df_model)
    return stat, chi2.sf(stat, ddf)


def _ols(y, X):
    return sm.OLS(y, sm.add_constant(X)).fit()


def _wls(y, X, w):
    return sm.WLS(y, sm.add_constant(X), weights=w).fit()


def fit_outcome(df, outcome):
    """Fit Baseline / Expanded(unscaled) / Expanded(scaled) / Weighted for one
    outcome on a single complete-case sample, and return result rows + N."""
    d = df.dropna(subset=[outcome] + COVARIATES + [CP, "num_clips"]).copy()
    n = len(d)

    # scaled outcome: z-scored over this outcome's complete-case sample (ddof=0)
    scaled_outcome = "__outcome_scaled__"
    d[scaled_outcome] = (d[outcome] - d[outcome].mean()) / d[outcome].std(ddof=0)

    w = d["num_clips"].astype(float).clip(lower=1)
    w = (w / w.mean()).values

    Xb, Xe = d[COVARIATES], d[COVARIATES + [CP]]

    ols_b, ols_e = _ols(d[outcome], Xb), _ols(d[outcome], Xe)          # unweighted
    wls_b, wls_e = _wls(d[outcome], Xb, w), _wls(d[outcome], Xe, w)     # weighted, raw outcome
    sca_b, sca_e = _wls(d[scaled_outcome], Xb, w), _wls(d[scaled_outcome], Xe, w)  # weighted, scaled outcome

    chi_u, p_u = _lrt(ols_b, ols_e)
    chi_w, p_w = _lrt(wls_b, wls_e)
    chi_s, p_s = _lrt(sca_b, sca_e)

    def row(model, fit, lrt=None, cp=True):
        r = {"Variable": outcome, "Model": model,
             "beta": fit.params[CP] if cp else np.nan,
             "SE": fit.bse[CP] if cp else np.nan,
             "p": fit.pvalues[CP] if cp else np.nan,
             "R2": fit.rsquared, "LogLik": fit.llf,
             "chi2": np.nan, "p_LRT": np.nan, "N": n}
        if lrt is not None:
            r["chi2"], r["p_LRT"] = lrt
        return r

    return [
        row("Baseline (unscaled)", ols_b, cp=False),
        row("Expanded (unscaled)", ols_e, lrt=(chi_u, p_u)),
        row("Expanded (scaled)",   sca_e, lrt=(chi_s, p_s)),
        row("Weighted",            wls_e, lrt=(chi_w, p_w)),
    ], n


def results_table(path, outcomes):
    df = load_master(path)
    rows = []
    for o in outcomes:
        r, _ = fit_outcome(df, o)
        rows += r
    out = pd.DataFrame(rows)[["Variable", "Model", "beta", "SE", "p",
                              "R2", "LogLik", "chi2", "p_LRT", "N"]]
    return out.round({"beta": 2, "SE": 2, "p": 4, "R2": 2,
                      "LogLik": 2, "chi2": 2, "p_LRT": 4})


In [ ]:
# --- Configure the input file, then run all cells ---
MASTER = "tp1_tp1_master_with_scaled.csv"      # path to the master CSV (edit as needed)
OUTCOMES = ['GFTA_Standard', 'EVT_Standard']

results = results_table(MASTER, OUTCOMES)
print("TABLE 4 (Timepoint 1 concurrent)")
results